# Phase 3 — Feature Engineering, continued: Age, Injury, OL Proxies, Draft Capital

This notebook picks up where `03_feature_investigation.ipynb` left off (Step 1: data-availability triage, Step 2: opportunity/efficiency/scarcity/trend features). Notebooks don't share kernel state, so the setup section below rebuilds `features_df` using the exact same already-validated logic from Step 2 before adding **Step 3: Age, Injury, OL Proxies, Draft Capital** — no re-derivation of reasoning already settled in `03_feature_investigation.ipynb`, just a quick, faithful rebuild.

In [1]:
from datetime import datetime

print(f"Results as of {datetime.now().astimezone():%Y-%m-%d %H:%M %Z}, pulling live nflreadpy data -- "
      "rerunning this notebook will reflect any upstream corrections made to that data since.")

Results as of 2026-09-15 11:30 Central Daylight Time, pulling live nflreadpy data -- rerunning this notebook will reflect any upstream corrections made to that data since.


## Setup: rebuild `features_df` from Steps 1–2

In [2]:
import sys
from pathlib import Path

import nflreadpy as nfl
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

vorp_labels = pd.read_parquet(REPO_ROOT / "data/processed/vorp_labels.parquet")

features_df = vorp_labels[[
    "season", "player_id", "player_display_name", "position", "recent_team", "vorp", "vorp_next",
    "target_share", "air_yards_share", "wopr",
    "passing_epa", "rushing_epa", "receiving_epa",
]].copy()

# 2c. Positional scarcity (scarcity_z + position_std_vorp_that_season)
season_position_stats = (
    features_df.groupby(["season", "position"])["vorp"]
    .agg(position_mean_vorp="mean", position_std_vorp_that_season="std")
    .reset_index()
)
features_df = features_df.merge(season_position_stats, on=["season", "position"], how="left")
features_df["scarcity_z"] = (
    (features_df["vorp"] - features_df["position_mean_vorp"]) / features_df["position_std_vorp_that_season"]
)
features_df = features_df.drop(columns=["position_mean_vorp"])

# 2d. Multi-year trend (vorp_delta_yoy)
prior_season_vorp = features_df[["player_id", "season", "vorp"]].copy()
prior_season_vorp["season"] = prior_season_vorp["season"] + 1
prior_season_vorp = prior_season_vorp.rename(columns={"vorp": "vorp_last_season"})
features_df = features_df.merge(prior_season_vorp, on=["player_id", "season"], how="left")
features_df["vorp_delta_yoy"] = features_df["vorp"] - features_df["vorp_last_season"]
features_df = features_df.drop(columns=["vorp_last_season"])

print(f"Rebuilt features_df: {len(features_df)} rows, {features_df.shape[1]} columns "
      f"(seasons {features_df['season'].min()}-{features_df['season'].max()})")

Rebuilt features_df: 11417 rows, 16 columns (seasons 2008-2025)


## Step 3: Age, Injury, OL Proxies, Draft Capital

QB starter-projection is explicitly skipped here, per scope.

### 3a. Age — `age` and `age_squared`

`birth_date` from `load_players()` (100% coverage, confirmed in Step 1). Age is computed as of **September 1st of the season** — a fixed, consistent reference point for "start of season" rather than each player's actual Week 1 game date, which varies. `age_squared` is added **alongside** `age`, not instead of it: production rises through a player's early-to-mid 20s and declines afterward, and a single linear `age` term can't represent that curve — a model needs the squared term to fit the shape rather than being forced through a straight line.

In [3]:
players = nfl.load_players().to_pandas()

features_df = features_df.merge(players[["gsis_id", "birth_date"]], left_on="player_id", right_on="gsis_id", how="left")
features_df = features_df.drop(columns=["gsis_id"])
features_df["birth_date"] = pd.to_datetime(features_df["birth_date"])

season_start = pd.to_datetime(features_df["season"].astype(str) + "-09-01")
features_df["age"] = (season_start - features_df["birth_date"]).dt.days / 365.25
features_df["age_squared"] = features_df["age"] ** 2
features_df = features_df.drop(columns=["birth_date"])

print(f"age: null rate {features_df['age'].isna().mean():.1%}, "
      f"range [{features_df['age'].min():.1f}, {features_df['age'].max():.1f}]")
print("(DEF rows are expected to be 100% null here — a team unit has no birth_date; "
      "age only applies to individual players.)")
print(features_df.groupby("position")["age"].apply(lambda s: s.isna().mean()))

age: null rate 4.8%, range [20.6, 46.7]
(DEF rows are expected to be 100% null here — a team unit has no birth_date; age only applies to individual players.)
position
DEF    1.0
K      0.0
QB     0.0
RB     0.0
TE     0.0
WR     0.0
Name: age, dtype: float64


### 3b. Injury — `injury_designations_count`

Built as a **concurrent-season** count (how many weeks *this* season a player carried a Questionable/Doubtful/Out designation), not a lagged "last season" feature — deliberately, for consistency with every other Step 2/3 feature so far (`target_share`, `wopr`, the EPA columns, `age`): they're all *this season's* information, used to predict *next season's* `vorp_next`. A concurrent injury count is not leakage here — it's fully known by the time the season it describes has ended, exactly like every other input feature.

**2008 rows are NaN, not 0** — confirmed in Step 1, `load_injuries()` has a hard floor at 2009 (the library itself rejects a 2008 request). A `0` would silently claim "this player was never injured in 2008," when the true answer is "we have no injury data for 2008 at all." **Any feature matrix that includes this column will need to explicitly drop or otherwise handle 2008 rows before training** — a `NaN` here means *unknown*, not *healthy*, and treating it as `0` would quietly bias the model toward thinking every 2008 player-season was injury-free.

In [4]:
injuries = nfl.load_injuries(seasons=True).to_pandas()
flagged = injuries[injuries["report_status"].isin(["Questionable", "Doubtful", "Out"])]
injury_counts = (
    flagged.groupby(["gsis_id", "season"])
    .size()
    .reset_index(name="injury_designations_count")
)

features_df = features_df.merge(
    injury_counts, left_on=["player_id", "season"], right_on=["gsis_id", "season"], how="left"
)
features_df = features_df.drop(columns=["gsis_id"])

# A player with zero Q/D/O designations that season is a real, known 0 (they were
# never flagged) -- EXCEPT for 2008, where load_injuries() has no data at all, so
# a post-merge NaN there means "unknown" and must NOT be filled with 0.
has_2009plus_data = features_df["season"] >= 2009
features_df.loc[has_2009plus_data, "injury_designations_count"] = (
    features_df.loc[has_2009plus_data, "injury_designations_count"].fillna(0)
)

print(f"injury_designations_count null rate: {features_df['injury_designations_count'].isna().mean():.1%}")
print("Null rate by season (2008 should be 100%, everything else near 0%):")
print(features_df.groupby("season")["injury_designations_count"].apply(lambda s: s.isna().mean()).to_string())

injury_designations_count null rate: 4.9%
Null rate by season (2008 should be 100%, everything else near 0%):
season
2008    1.0
2009    0.0
2010    0.0
2011    0.0
2012    0.0
2013    0.0
2014    0.0
2015    0.0
2016    0.0
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    0.0
2025    0.0


### 3c. OL proxies — `ol_pass_protection_proxy`, `ol_run_blocking_proxy`

Rebuilt with the **exact same formulas** already verified in `03_feature_investigation.ipynb` (sack rate allowed from `load_team_stats()`, stuff rate excluding QB scrambles from `load_pbp()`, both 2008-2025) -- no re-derivation, just brought over.

Joining these onto `features_df` at the team-season level surfaced one real issue, checked and handled rather than assumed away:

**Franchise relocation codes differ by data source.** `recent_team` (player-level, from `load_players()`/seasonal stats) uses each team's **current** abbreviation retroactively -- Derek Carr's `recent_team` is `"LV"` for every season back to 2014, even the ones he played in Oakland. Directly checked `load_team_stats()`/`load_pbp()` the same way, and they **also** use current codes (`"LV"`, `"LAC"`, `"LA"`) for old seasons -- so offensive rows need no translation at all; they already match. **DEF rows are the one exception**: Phase 2 built those from the *Sleeper* API, which -- confirmed back in Phase 2 -- uses **era-accurate** codes (`OAK`/`SD`/`STL`/`LAR`), a real and permanent historical-accuracy difference, not a bug. A small map (`OAK->LV`, `SD->LAC`, `STL->LA`, `LAR->LA`) reconciles DEF's team key to the nflreadpy convention before joining.

*(Previously, this section also had to drop 4 known-bad DEF rows -- a Raiders OAK/LV duplicate for 2017-2019 and one garbage "1339z" row -- discovered while building this join. That was a real root-cause bug in Phase 2's `get_stats_week` ingestion: Sleeper stubs a franchise's data under its FUTURE code years before a relocation actually happens, and a stray non-numeric player id slipped past the old team-code filter. Both are now fixed at the source in `02_vorp_target_construction.ipynb` / `sleeper_client.py` -- confirmed directly against Sleeper's raw API response, not guessed -- with a permanent regression-guard assertion added there. `vorp_labels.parquet` is clean as of the latest Phase 2 rebuild, so the manual drop is no longer needed here.)*

In [5]:
# --- Team key normalization: DEF's era-accurate Sleeper codes -> nflreadpy's current-code convention ---
# (Still needed: this is a real, permanent historical-accuracy difference between
# the two data sources, not the OAK/LV duplicate bug -- that was fixed at the
# source in Phase 2, so no manual row-dropping is needed here anymore.)
DEF_TEAM_CODE_NORMALIZE = {"OAK": "LV", "SD": "LAC", "STL": "LA", "LAR": "LA"}
features_df["team_for_ol_join"] = features_df["recent_team"]
def_mask = features_df["position"] == "DEF"
features_df.loc[def_mask, "team_for_ol_join"] = (
    features_df.loc[def_mask, "player_id"].map(DEF_TEAM_CODE_NORMALIZE).fillna(features_df.loc[def_mask, "player_id"])
)

# --- ol_pass_protection_proxy (exact formula from 03_feature_investigation.ipynb) ---
team_stats = nfl.load_team_stats(seasons=True, summary_level="reg").to_pandas()
team_stats_range = team_stats[(team_stats["season"] >= 2008) & (team_stats["season"] <= 2025)].copy()
team_stats_range["ol_pass_protection_proxy"] = (
    team_stats_range["sacks_suffered"] / (team_stats_range["attempts"] + team_stats_range["sacks_suffered"])
)

features_df = features_df.merge(
    team_stats_range[["season", "team", "ol_pass_protection_proxy"]],
    left_on=["season", "team_for_ol_join"], right_on=["season", "team"], how="left",
)
features_df = features_df.drop(columns=["team"])

print(f"ol_pass_protection_proxy null rate: {features_df['ol_pass_protection_proxy'].isna().mean():.1%}")

ol_pass_protection_proxy null rate: 0.0%


In [6]:
# --- ol_run_blocking_proxy (exact formula from 03_feature_investigation.ipynb) ---
EARLIEST_SEASON = 2008
LATEST_SEASON = 2025

stuff_rows = []
for season in range(EARLIEST_SEASON, LATEST_SEASON + 1):
    pbp_season = nfl.load_pbp(seasons=[season]).to_pandas()
    runs = pbp_season[(pbp_season["play_type"] == "run") & (pbp_season["qb_scramble"] != 1)]
    runs = runs.dropna(subset=["rushing_yards", "posteam"])

    season_summary = runs.groupby("posteam")["rushing_yards"].agg(
        attempts="size", stuffed=lambda s: (s <= 0).sum()
    )
    season_summary["season"] = season
    stuff_rows.append(season_summary.reset_index().rename(columns={"posteam": "team"}))
    del pbp_season

    print(f"  {season}: done ({len(runs)} qualifying run plays)")

stuff_rate_by_team_season = pd.concat(stuff_rows, ignore_index=True)
stuff_rate_by_team_season["ol_run_blocking_proxy"] = (
    stuff_rate_by_team_season["stuffed"] / stuff_rate_by_team_season["attempts"]
)

features_df = features_df.merge(
    stuff_rate_by_team_season[["season", "team", "ol_run_blocking_proxy"]],
    left_on=["season", "team_for_ol_join"], right_on=["season", "team"], how="left",
)
features_df = features_df.drop(columns=["team", "team_for_ol_join"])

print(f"\nol_run_blocking_proxy null rate: {features_df['ol_run_blocking_proxy'].isna().mean():.1%}")
print("Both OL proxies null rate by position (should be 0% everywhere -- every team-season matched):")
print(features_df.groupby("position")[["ol_pass_protection_proxy", "ol_run_blocking_proxy"]].apply(lambda g: g.isna().mean()))

  2008: done (13752 qualifying run plays)


  2009: done (13782 qualifying run plays)


  2010: done (13462 qualifying run plays)


  2011: done (13443 qualifying run plays)


  2012: done (13481 qualifying run plays)


...

  2021: done (13760 qualifying run plays)


  2022: done (14046 qualifying run plays)


  2023: done (13734 qualifying run plays)


  2024: done (13872 qualifying run plays)


  2025: done (13714 qualifying run plays)

ol_run_blocking_proxy null rate: 0.0%
Both OL proxies null rate by position (should be 0% everywhere -- every team-season matched):
          ol_pass_protection_proxy  ol_run_blocking_proxy
position                                                 
DEF                            0.0                    0.0
K                              0.0                    0.0
QB                             0.0                    0.0
RB                             0.0                    0.0
TE                             0.0                    0.0
WR                             0.0                    0.0


### 3d. Draft capital — `draft_capital_tier` and `draft_pick_inverse`

Raw pick number isn't used directly — pick 1 isn't linearly "32x more valuable" than pick 32; the value curve is steep early and flattens out fast. Two transforms, covering both options from the roadmap:

- **`draft_capital_tier`** (categorical): `"Round 1"`, `"Round 2-3"`, `"Round 4+"` (covers round 4 through the rare pre-1994 12-round-era stragglers who still show up a few times in this data), and **`"UDFA"`** as its own explicit category — not a numeric placeholder that could be misread as an actual late-round pick.
- **`draft_pick_inverse`** (continuous): `1 / draft_pick` for drafted players, capturing the steep early dropoff a tier can't. **`NaN` for real UDFAs** — not `0` and not some sentinel pick number, since either would misrepresent "never drafted" as a numeric draft position.

In [7]:
features_df = features_df.merge(
    players[["gsis_id", "draft_round", "draft_pick"]], left_on="player_id", right_on="gsis_id", how="left"
)
features_df = features_df.drop(columns=["gsis_id"])


def draft_tier(draft_round):
    if pd.isna(draft_round):
        return "UDFA"
    if draft_round == 1:
        return "Round 1"
    if draft_round in (2, 3):
        return "Round 2-3"
    return "Round 4+"


features_df["draft_capital_tier"] = features_df["draft_round"].apply(draft_tier)
features_df["draft_pick_inverse"] = 1 / features_df["draft_pick"]  # NaN propagates naturally for UDFA (draft_pick is NaN)
features_df = features_df.drop(columns=["draft_round", "draft_pick"])

print("draft_capital_tier value counts (offense only -- DEF has no individual draft record):")
print(features_df[features_df["position"] != "DEF"]["draft_capital_tier"].value_counts().to_string())
print()
print(f"draft_pick_inverse null rate: {features_df['draft_pick_inverse'].isna().mean():.1%} "
      f"(should equal the UDFA share above, plus 100% of DEF rows)")

draft_capital_tier value counts (offense only -- DEF has no individual draft record):
draft_capital_tier
Round 4+     3444
UDFA         3349
Round 2-3    2495
Round 1      1585

draft_pick_inverse null rate: 34.1% (should equal the UDFA share above, plus 100% of DEF rows)


## Full null-rate summary: Step 2 + Step 3 features together

In [8]:
all_new_features = [
    # Step 2
    "target_share", "air_yards_share", "wopr",
    "passing_epa", "rushing_epa", "receiving_epa",
    "scarcity_z", "position_std_vorp_that_season", "vorp_delta_yoy",
    # Step 3
    "age", "age_squared", "injury_designations_count",
    "ol_pass_protection_proxy", "ol_run_blocking_proxy",
    "draft_capital_tier", "draft_pick_inverse",
]

null_summary = pd.DataFrame({
    "feature": all_new_features,
    "null_rate": [f"{features_df[c].isna().mean():.1%}" for c in all_new_features],
})
print(f"features_df: {len(features_df)} rows, {features_df.shape[1]} columns\n")
print(null_summary.to_string(index=False))

features_df: 11417 rows, 23 columns

                      feature null_rate
                 target_share      7.7%
              air_yards_share      4.8%
                         wopr      7.7%
                  passing_epa     84.8%
                  rushing_epa     52.2%
                receiving_epa     27.5%
                   scarcity_z      0.0%
position_std_vorp_that_season      0.0%
               vorp_delta_yoy     28.6%
                          age      4.8%
                  age_squared      4.8%
    injury_designations_count      4.9%
     ol_pass_protection_proxy      0.0%
        ol_run_blocking_proxy      0.0%
           draft_capital_tier      0.0%
           draft_pick_inverse     34.1%
